In [0]:
from pyspark.sql.functions import *

In [0]:
base_adls2_path = "abfss://customer360unitycatalog@kmstorage9490.dfs.core.windows.net"

In [0]:
df = spark.table('customer360_cata.bronze.orders')

In [0]:
df = df.withColumn(
    "order_date",
    coalesce(
        try_to_date(col("order_date"), "yyyy/MM/dd"),
        try_to_date(col("order_date"), "dd-MM-yyyy"),
        try_to_date(col("order_date"), "yyyy-MM-dd"),
        try_to_date(col("order_date"), "yyyyMMdd"),
        try_to_date(col("order_date"), "MM-dd-yyyy"),
        try_to_date(col("order_date"), "dd/MM/yyyy"),
    )
)

In [0]:
df = df.withColumn('status',initcap(col("status")))

In [0]:
df = df.withColumn("amount",col("amount").cast("double"))
df = df.withColumn('amount',when(col('amount').isNull(),0).when(col('amount')<0,0).otherwise(col('amount')))

In [0]:
df =df.dropDuplicates(["order_id"])
df = df.dropna(subset=["order_id","customer_id","order_date"])

In [0]:
df = df.withColumn("customer_id", col("customer_id").cast("int")).withColumn("order_id", col("order_id").cast("int"))

In [0]:
df =df.drop("_rescued_data")

In [0]:
df.write.format("delta").mode("overwrite")\
    .option('overwriteSchema', 'true') \
    .option('path',base_adls2_path + '/silver/orders') \
        .saveAsTable('customer360_cata.silver.orders')